In [ ]:
!wget http://www.agentspace.org/download/watch4u.zip
!unzip -o watch4u.zip
!rm -f watch4u.zip

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
model = YOLO("yolov3u.pt")

In [ ]:
results = model.train(data="watch.yaml", epochs=30, imgsz=416)

In [ ]:
results = model("datasets/watch/val/000011.jpg", save=True)

In [ ]:
import torch
torch.save(model.model,'watch_detector_yolo3u.pth')

In [ ]:
from google.colab import files
files.download('watch_detector_yolo3u.pth')

In [ ]:
model.model

In [ ]:
example = torch.randn(1, 3, 416, 416).cuda()
traced = torch.jit.trace(model.model, example)
traced.save("watch_detector_yolo3u_traced.pth")

In [ ]:
from google.colab import files
files.download('watch_detector_yolo3u_traced.pth')

In [ ]:
import cv2
import numpy as np

In [ ]:
def preprocess(img, img_size=416, border_color=114):
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # convert to RGB

    # Resize with unchanged aspect ratio (letterbox)
    h, w = img.shape[:2]
    scale = min(img_size / h, img_size / w)
    nh, nw = int(h * scale), int(w * scale)
    img_resized = cv2.resize(img, (nw, nh))
    new_img = np.full((img_size, img_size, 3), border_color, dtype=np.uint8)
    new_img[:nh, :nw] = img_resized

    new_img = new_img.astype(np.float32) / 255.0 # normalize into 0-1
    new_img = np.transpose(new_img, (2, 0, 1)) # convert H,W,C to C,H,W
    tensor = torch.from_numpy(new_img).unsqueeze(0) # turn to tensor B,C,H,W
    return tensor, scale, nw, nh

In [ ]:
from ultralytics.utils.nms import non_max_suppression

def postprocess(predictions, orig, scale, nw, nh):
    pred = non_max_suppression(predictions, conf_thres=0.75, iou_thres=0.45)[0]
    oh, ow = orig[:2]
    pred[:, [0,2]] = pred[:, [0,2]] / nw * ow
    pred[:, [1,3]] = pred[:, [1,3]] / nh * oh
    return pred

In [ ]:
img = cv2.imread("image.jpg")
x, scale, nw, nh = preprocess(img)
x = x.to("cuda")
y = model.model(x)
y = y[0].cpu()
pred = postprocess(y, img.shape, scale, nw, nh)

In [ ]:
pred

In [ ]:
def plot_one_box(xyxy, img, label=None, color=(0,255,0), line_thickness=None): # Plots one bounding box on the image
    # Calculate line thickness based on image size
    tl = line_thickness or round(0.002 * (img.shape[0] + img.shape[1]) / 2) + 1
    # Convert bounding box coordinates to integers
    pt1, pt2 = (int(xyxy[0]), int(xyxy[1])), (int(xyxy[2]), int(xyxy[3]))
    # Draw the bounding box rectangle on the image
    cv2.rectangle(img, pt1, pt2, color, thickness=tl)
    if label:
        # Calculate font thickness
        tf = max(tl - 1, 1)
        # Calculate text size
        t_size = cv2.getTextSize(label, 0, fontScale=tl / 3, thickness=tf)[0]
        # Calculate the coordinates for the label background rectangle
        pt3 = pt1[0] + t_size[0], pt1[1] - t_size[1] - 3
        # Draw the label background rectangle
        cv2.rectangle(img, pt1, pt3, color, cv2.FILLED)
        # Draw the label text
        cv2.putText(img, label, (pt1[0], pt1[1] - 2), 0, tl / 3, (0,0,0), thickness=tf)

In [ ]:
names=['watch']

In [ ]:
disp = np.copy(img)
for detection in pred:
    *xyxy, confidence, classid  = detection
    print(f'{confidence.item():.2f},{int(xyxy[0].item())},{int(xyxy[1].item())},{int(xyxy[2].item())},{int(xyxy[3].item())},{names[classid.int().item()]}')
    plot_one_box(xyxy, disp, label=names[classid.int().item()], color=(0,255,0))


In [ ]:
#display result
from google.colab.patches import cv2_imshow
cv2_imshow(disp)